## Pobranie danych o 50 gwiazdach

In [9]:
from astroquery.gaia import Gaia
from astropy import units as u
from astropy.table import Table

# Upewniamy się, że korzystamy z odpowiedniej tabeli DR3
Gaia.MAIN_GAIA_TABLE = "gaiadr3.gaia_source"

# Zapytanie ADQL:
# - bierzemy 50 obiektów z największą paralaksą (parallax w mas)
# - pomijamy parallax <= 0, bo to bez sensu fizycznie
query = """
SELECT
    TOP 50
    g.source_id,
    g.ra, g.dec,
    g.parallax,                     -- w mili-arcsekundach
    g.pmra, g.pmdec,                -- własne ruchy [mas/yr]
    g.radial_velocity,              -- km/s (często NULL)
    g.phot_g_mean_mag,
    ap.mass_flame,                  -- masa gwiazdy [Msun] (modelowana)
    ap.radius_flame                 -- promień [Rsun] (modelowany)
FROM gaiadr3.gaia_source AS g
LEFT JOIN gaiadr3.astrophysical_parameters AS ap
  ON g.source_id = ap.source_id
WHERE g.parallax > 0
ORDER BY g.parallax DESC
"""

job = Gaia.launch_job_async(query)
tbl = job.get_results()

# Dodajemy kolumnę z odległością w parsekach: d[pc] = 1000 / parallax[mas]
distance_pc = (1000.0 / tbl["parallax"]) * u.pc
tbl["distance_pc"] = distance_pc

# Możesz obejrzeć pierwsze wiersze
tbl[:5].pprint(max_width=120)

# Zapis do CSV, np. do dalszej obróbki / REBOUND
tbl.write("nearest_50_gaia.csv", format="csv", overwrite=True)
print("Zapisano do nearest_50_gaia.csv")

INFO: Query finished. [astroquery.utils.tap.core]
     source_id              ra                 dec         ... mass_flame radius_flame    distance_pc    
                           deg                 deg         ...  solMass      solRad            pc        
------------------- ------------------ ------------------- ... ---------- ------------ ------------------
5853498713190525696 217.39232147200883  -62.67607511676666 ...         --   0.16237111 1.3019705311704333
4472832130942575872 269.44850252543836   4.739420051112412 ...         --   0.20216417  1.828233981355542
3864972938605115520 164.10319030755974   7.002726940984864 ...         --           --  2.408597252748958
 762815470562110464 165.83095967577933  35.948653032660104 ...         --   0.41768724 2.5461298549469777
2947050466531873024 101.28662552099249 -16.720932526023173 ...         --           -- 2.6703012063129745
Zapisano do nearest_50_gaia.csv


## Tworzenie układu słonecznego, nadanie mas i rozmiarów

## Sprawdzenie czy wszystko się zgadza

In [ ]:
import numpy as np
import pandas as pd
from astropy.table import Table

# 3. Podsumowanie: położenie, masa i rozmiar wszystkich ciał w symulacji
# Zakładamy, że komórka 0 (Gaia) i 1 (REBOUND+SPICE) zostały już wykonane

# Wczytujemy tabelę gwiazd, żeby mieć ich identyfikatory
stars_tbl = Table.read("nearest_50_gaia.csv", format="csv")

rows = []

# Indeksy w REBOUND: 0 = Słońce, 1–8 = planety, 9+ = gwiazdy

# Słońce
p0 = sim.particles[0]
rows.append({
    "name": "SUN",
    "kind": "star_sun",
    "mass_Msun": p0.m,
    "radius_AU": p0.r,
    "x_AU": p0.x,
    "y_AU": p0.y,
    "z_AU": p0.z,
})

# Planety
for i, body in enumerate(solar_system_bodies, start=1):
    p = sim.particles[i]
    rows.append({
        "name": body,
        "kind": "planet",
        "mass_Msun": p.m,
        "radius_AU": p.r,
        "x_AU": p.x,
        "y_AU": p.y,
        "z_AU": p.z,
    })

# Gwiazdy z Gaia
offset = 1 + len(solar_system_bodies)
for j in range(len(stars_tbl)):
    p = sim.particles[offset + j]
    sid = int(stars_tbl["source_id"][j])
    rows.append({
        "name": f"GAIA_DR3_{sid}",
        "kind": "star_gaia",
        "mass_Msun": p.m,
        "radius_AU": p.r,
        "x_AU": p.x,
        "y_AU": p.y,
        "z_AU": p.z,
    })

bodies_df = pd.DataFrame(rows)

# Podgląd tabeli
print(bodies_df.head(12))
print("\nPodsumowanie typów:")
print(bodies_df["kind"].value_counts())

                            name       kind     mass_Msun  radius_AU  \
0                            SUN   star_sun  1.000000e+00   0.004650   
1                        Mercury     planet  1.660121e-07   0.000016   
2                          Venus     planet  2.447838e-06   0.000041   
3                          Earth     planet  3.040433e-06   0.000043   
4                           Mars     planet  3.227156e-07   0.000023   
5                        Jupiter     planet  9.547919e-04   0.000478   
6                         Saturn     planet  2.858857e-04   0.000403   
7                         Uranus     planet  4.366250e-05   0.000171   
8                        Neptune     planet  5.151384e-05   0.000165   
9   GAIA_DR3_5853498713190525696  star_gaia  1.002175e-01   0.000755   
10  GAIA_DR3_4472832130942575872  star_gaia  1.498180e-01   0.000940   
11  GAIA_DR3_3864972938605115520  star_gaia  8.296717e-02   0.000635   

             x_AU           y_AU           z_AU  
0       -0.00

## Przejście do układu barycentrycznego

In [9]:
import numpy as np

# 4. Przejście do układu barycentrycznego (opcjonalne, ale zalecane)
# Zakładamy, że komórka 1 została wykonana i obiekt `sim` zawiera
# już Słońce, planety i 50 gwiazd z Gaia.

# Barycentrum i całkowity pęd PRZED transformacją (dla ciekawości)
M_tot = 0.0
R_cm = np.zeros(3)
P_tot = np.zeros(3)

for p in sim.particles:
    m = p.m
    r_vec = np.array([p.x, p.y, p.z])
    v_vec = np.array([p.vx, p.vy, p.vz])
    M_tot += m
    R_cm += m * r_vec
    P_tot += m * v_vec

R_cm /= M_tot
V_cm = P_tot / M_tot

print("Przed move_to_com():")
print(f"  R_cm = {R_cm} AU")
print(f"  V_cm = {V_cm} AU/yr")

# Przesunięcie do układu barycentrycznego
sim.move_to_com()

# Barycentrum i pęd PO transformacji (powinny być ~0)
M_tot2 = 0.0
R_cm2 = np.zeros(3)
P_tot2 = np.zeros(3)

for p in sim.particles:
    m = p.m
    r_vec = np.array([p.x, p.y, p.z])
    v_vec = np.array([p.vx, p.vy, p.vz])
    M_tot2 += m
    R_cm2 += m * r_vec
    P_tot2 += m * v_vec

R_cm2 /= M_tot2
V_cm2 = P_tot2 / M_tot2

print("\nPo move_to_com():")
print(f"  R_cm = {R_cm2} AU")
print(f"  V_cm = {V_cm2} AU/yr")

Przed move_to_com():
  R_cm = [151189.71871984  -1653.33689666 -15319.84057676] AU
  V_cm = [-0.94619316  6.73417651 -2.49309944] AU/yr

Po move_to_com():
  R_cm = [ 4.17820377e-11 -1.19377251e-11  0.00000000e+00] AU
  V_cm = [9.90469055e-17 1.23865555e-15 1.45724183e-16] AU/yr


## funkcje na parametr beta dla Ciśnienia promieniowania

In [36]:
import numpy as np
from astropy.constants import G as G_SI, M_sun as M_SUN_SI, L_sun as L_SUN_SI, c as C_SI
import astropy.units as u

# 5. Funkcje pomocnicze do ciśnienia promieniowania
# -------------------------------------------------
# Założenia :
# - ciśnienie promieniowania działa TYLKO na asteroidy (cząstki testowe o małych masach),
# - w danym kroku liczymy wkład TYLKO od najbliższej gwiazdy (Słońce lub gwiazdy z Gaia),
# - jasność gwiazdy L_* szacujemy z masy: L/L_sun ~ (M/M_sun)^3.5.


def compute_beta_single_star(M_star_Msun, R_body_m, rho=2000.0, Q_pr=1.0):
    """Zwraca współczynnik beta = Frad/Fgrav dla małego ciała przy gwieździe.

    Obsługuje zarówno skalary, jak i tablice numpy (R_body_m, rho, Q_pr
    mogą być tablicami — wtedy wynik też jest tablicą).

    Parametry
    ---------
    M_star_Msun : float — masa gwiazdy [Msun]
    R_body_m    : float | array — promień ciała (asteroidy) [m]
    rho         : float | array — gęstość ciała [kg/m^3]
    Q_pr        : float | array — współczynnik efektywności ciśnienia promieniowania

    Zwraca
    -------
    beta : float | array — bezwymiarowy stosunek siły promieniowania do grawitacji.
    """
    L_star_Lsun = M_star_Msun ** 3.5

    L_star_SI = L_star_Lsun * L_SUN_SI.value
    M_star_SI = M_star_Msun * M_SUN_SI.value

    beta = (3.0 * L_star_SI * Q_pr) / (16.0 * np.pi * C_SI.value * G_SI.value * M_star_SI * rho * R_body_m)
    return beta


def radiation_pressure_accel_nearest_star(
    sim,
    i_ast: int,
    star_indices,
    R_body_m: float,
    rho: float = 2000.0,
    Q_pr: float = 1.0,
):
    """Liczy przyspieszenie od ciśnienia promieniowania dla wybranej asteroidy.

    - Szuka najbliższej gwiazdy z listy star_indices (np. [0] + indeksy gwiazd Gaia).
    - Dla tej gwiazdy liczy beta (Frad/Fgrav) z prostego wzoru.
    - Z beta i grawitacji tej gwiazdy liczy wektor przyspieszenia od promieniowania.

    Zwraca
    -------
    a_rad_vec : np.array(3) w jednostkach REBOUND (AU/yr^2)
    beta      : bezwymiarowy współczynnik ciśnienia promieniowania dla tej asteroidy
    j_near    : indeks najbliższej gwiazdy w sim.particles
    """
    p_ast = sim.particles[i_ast]

    # Szukamy najbliższej gwiazdy
    r_min = np.inf
    j_near = None
    for j in star_indices:
        p_star = sim.particles[j]
        dx = p_ast.x - p_star.x
        dy = p_ast.y - p_star.y
        dz = p_ast.z - p_star.z
        r2 = dx * dx + dy * dy + dz * dz
        if r2 < r_min:
            r_min = r2
            j_near = j

    if j_near is None:
        # Brak gwiazd na liście – brak ciśnienia promieniowania
        return np.zeros(3), 0.0, None

    p_star = sim.particles[j_near]
    r_vec_AU = np.array([p_ast.x - p_star.x, p_ast.y - p_star.y, p_ast.z - p_star.z])
    r_AU = np.linalg.norm(r_vec_AU)

    # Beta dla tej gwiazdy i tej asteroidy (obliczone w SI, niezależne od odległości)
    M_star_Msun = p_star.m
    beta = compute_beta_single_star(M_star_Msun, R_body_m, rho=rho, Q_pr=Q_pr)

    # Grawitacyjne przyspieszenie od tej gwiazdy w jednostkach REBOUND (AU/yr^2)
    # REBOUND używa G=1 w jednostkach (AU, yr, Msun), więc a_grav = -m_star * r_vec / r^3.
    if r_AU == 0.0:
        return np.zeros(3), beta, j_near

    a_grav_vec = -M_star_Msun * r_vec_AU / (r_AU ** 3)

    # Ciśnienie promieniowania działa przeciwnie do grawitacji, z modułem beta * |a_grav|
    a_rad_vec = -beta * a_grav_vec  # znak minus: promieniowanie "odpycha" od gwiazdy

    return a_rad_vec, beta, j_near

## Funkcja tworząca wybuch (na razie na marsie)

In [37]:
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.constants import M_sun as M_SUN


# ===============================================================
#  Pomocnicze funkcje losowania
# ===============================================================

def _sample_truncated_power_law(x_min, x_max, alpha, n, rng):
    """Losuje n próbek z obciętego rozkładu potęgowego p(x) ∝ x^{-alpha}
    na przedziale [x_min, x_max] metodą odwrotnej CDF."""
    u_rand = rng.uniform(0.0, 1.0, n)
    if abs(alpha - 1.0) < 1e-10:
        return x_min * (x_max / x_min) ** u_rand
    a = 1.0 - alpha
    return (u_rand * (x_max**a - x_min**a) + x_min**a) ** (1.0 / a)


def _random_cone_directions(normal, half_angle_deg, n, rng):
    """Generuje n losowych wektorów jednostkowych w stożku o osi `normal`
    i kącie połówkowym `half_angle_deg`, z jednorodnym rozkładem na czapce sferycznej."""
    half_rad = np.radians(half_angle_deg)
    cos_theta = rng.uniform(np.cos(half_rad), 1.0, n)
    sin_theta = np.sqrt(1.0 - cos_theta**2)
    phi = rng.uniform(0.0, 2.0 * np.pi, n)

    local = np.column_stack([
        sin_theta * np.cos(phi),
        sin_theta * np.sin(phi),
        cos_theta,
    ])

    normal = np.asarray(normal, dtype=float)
    normal /= np.linalg.norm(normal)
    z_axis = np.array([0.0, 0.0, 1.0])

    if np.allclose(normal, z_axis):
        return local
    if np.allclose(normal, -z_axis):
        local[:, 2] *= -1
        return local

    # Macierz obrotu (Rodrigues) z osi z na wektor `normal`
    v = np.cross(z_axis, normal)
    s = np.linalg.norm(v)
    c_val = np.dot(z_axis, normal)
    V = np.array([[0, -v[2], v[1]],
                   [v[2], 0, -v[0]],
                   [-v[1], v[0], 0]])
    R_mat = np.eye(3) + V + V @ V * (1.0 - c_val) / (s**2)
    return (R_mat @ local.T).T


# ===============================================================
#  Domyślne warianty skał (tutaj dodamy nasze!!!)
# ===============================================================
DEFAULT_ROCK_VARIANTS = [
    {"name": "basalt",    "density": 3000.0, "albedo": 0.07, "prob": 0.50},
    {"name": "chondrite", "density": 3500.0, "albedo": 0.15, "prob": 0.30},
    {"name": "ice_rich",  "density": 1500.0, "albedo": 0.40, "prob": 0.20},
]


# ===============================================================
#  Główna funkcja: create_mars_impact
# ===============================================================

def create_mars_impact(
    sim,
    n_asteroids=100,
    # ---- Geometria uderzenia ----
    impact_normal=None,
    cone_half_angle=60.0,
    # ---- Prędkość ejecta ----
    v_min_kms=5.03,
    v_max_kms=20.0,
    alpha_v=2.5,
    # ---- Rozmiar ----
    R_min_m=0.001,
    R_max_m=5.0,
    q_size=2,
    # ---- Gęstość / typy skał ----
    rock_variants=None,
    # ---- Rotacja ----
    spin_period_range=(2.0, 20.0),
    obliquity_range=(0.0, 180.0),
    # ---- Korelacja rozmiar–prędkość ----
    size_velocity_corr=True,
    # ---- Indeksy gwiazd (źródła promieniowania) ----
    star_indices=None,
    # ---- Indeks Marsa w symulacji ----
    mars_index=4,
    # ---- Ziarno losowości ----
    seed=None,
):
    """
    Tworzy zdarzenie impaktowe na Marsie i dodaje asteroidy (ejecta) do symulacji.

    Parametry
    ---------
    sim               : rebound.Simulation
    n_asteroids       : int — liczba asteroid do wygenerowania
    impact_normal     : (3,) array | None — oś stożka ejecta; None = losowy kierunek
    cone_half_angle   : float [deg] — kąt połówkowy stożka ejecta
    v_min_kms         : float [km/s] — minimalna prędkość (domyślnie ~v_esc Marsa)
    v_max_kms         : float [km/s] — maksymalna prędkość ejecta
    alpha_v           : float — wykładnik potęgowy rozkładu prędkości (>1)
    R_min_m, R_max_m  : float [m] — zakres promieni asteroid
    q_size            : float — wykładnik potęgowy rozkładu rozmiarów (>1)
    rock_variants     : list[dict] — warianty skał {"name","density","albedo","prob"}
    spin_period_range : (min, max) [h] — zakres okresów rotacji
    obliquity_range   : (min, max) [deg] — zakres nachylenia osi obrotu
    size_velocity_corr: bool — antykorelacja rozmiar↔prędkość
    star_indices      : list[int] | None — indeksy gwiazd w sim.particles
                        (źródła promieniowania, w tym Słońce!). None = [0] (samo Słońce).
                        Aby uwzględnić gwiazdy Gaia: [0] + list(range(9, 59))
    mars_index        : int — indeks Marsa w sim.particles
    seed              : int | None — ziarno generatora losowego

    Zwraca
    -------
    asteroid_df : pd.DataFrame — metadane każdej asteroidy (do ATM i dalszej analizy)
    first_index : int — indeks pierwszej asteroidy w sim.particles
    """
    rng = np.random.default_rng(seed)

    if rock_variants is None:
        rock_variants = DEFAULT_ROCK_VARIANTS
    if star_indices is None:
        star_indices = [0]

    # ── Stan Marsa w chwili wybuchu ──────────────────────────────
    p_mars = sim.particles[mars_index]
    mars_pos = np.array([p_mars.x, p_mars.y, p_mars.z])
    mars_vel = np.array([p_mars.vx, p_mars.vy, p_mars.vz])

    # ── Normalna powierzchni w punkcie uderzenia ────────────────
    if impact_normal is None:
        vec = rng.standard_normal(3)
        impact_normal = vec / np.linalg.norm(vec)
    else:
        impact_normal = np.asarray(impact_normal, dtype=float)
        impact_normal /= np.linalg.norm(impact_normal)

    # ── 1. Losowanie promieni  (rozkład potęgowy) ───────────────
    radii_m = _sample_truncated_power_law(R_min_m, R_max_m, q_size, n_asteroids, rng)

    # ── 2. Losowanie prędkości ejecta (rozkład potęgowy) ────────
    velocities_kms = _sample_truncated_power_law(
        v_min_kms, v_max_kms, alpha_v, n_asteroids, rng,
    )

    # ── 3. Antykorelacja: mniejsze → szybsze ───────────────────
    if size_velocity_corr:
        sorted_r = np.sort(radii_m)
        sorted_v = np.sort(velocities_kms)[::-1]
        perm = rng.permutation(n_asteroids)
        radii_m = sorted_r[perm]
        velocities_kms = sorted_v[perm]

    # ── 4. Losowanie typu skały ─────────────────────────────────
    names   = [rv["name"]    for rv in rock_variants]
    dens    = np.array([rv["density"] for rv in rock_variants])
    albedos = np.array([rv["albedo"]  for rv in rock_variants])
    probs   = np.array([rv["prob"]    for rv in rock_variants])
    probs  /= probs.sum()

    idx = rng.choice(len(rock_variants), size=n_asteroids, p=probs)
    rho_arr    = dens[idx]
    albedo_arr = albedos[idx]
    rock_names = [names[i] for i in idx]

    # ── 5. Masa z geometrii: m = (4/3)πR³ρ ─────────────────────
    mass_kg   = (4.0 / 3.0) * np.pi * radii_m**3 * rho_arr
    mass_Msun = mass_kg / M_SUN.value

    # ── 6. Beta — ciśnienie promieniowania od najbliższej gwiazdy ──
    Q_pr_arr = 1.0 + (2.0 / 3.0) * albedo_arr

    # Szukamy najbliższej gwiazdy do pozycji Marsa z listy star_indices
    nearest_star_idx = star_indices[0]
    r2_min = np.inf
    for j in star_indices:
        ps = sim.particles[j]
        dx = mars_pos[0] - ps.x
        dy = mars_pos[1] - ps.y
        dz = mars_pos[2] - ps.z
        r2 = dx*dx + dy*dy + dz*dz
        if r2 < r2_min:
            r2_min = r2
            nearest_star_idx = j

    nearest_star_mass = sim.particles[nearest_star_idx].m
    beta_arr = compute_beta_single_star(
        nearest_star_mass, radii_m, rho=rho_arr, Q_pr=Q_pr_arr,
    )

    # ── 7. Kierunki ejecta (stożek wokół normalnej) ────────────
    directions = _random_cone_directions(
        impact_normal, cone_half_angle, n_asteroids, rng,
    )

    # ── 8. Prędkości w AU/yr = ejecta + prędkość orbitalna Marsa
    kms_to_au_yr = (1.0 * u.km / u.s).to(u.AU / u.yr).value
    v_au_yr = velocities_kms * kms_to_au_yr

    vx = directions[:, 0] * v_au_yr + mars_vel[0]
    vy = directions[:, 1] * v_au_yr + mars_vel[1]
    vz = directions[:, 2] * v_au_yr + mars_vel[2]

    # ── 9. Parametry rotacji ────────────────────────────────────
    spin_periods_h = rng.uniform(
        spin_period_range[0], spin_period_range[1], n_asteroids,
    )
    obliquities_deg = rng.uniform(
        obliquity_range[0], obliquity_range[1], n_asteroids,
    )
    spin_axes = rng.standard_normal((n_asteroids, 3))
    spin_axes /= np.linalg.norm(spin_axes, axis=1, keepdims=True)

    # ── 10. Dodanie cząstek do REBOUND ──────────────────────────
    m_to_au   = (1.0 * u.m).to(u.AU).value
    radii_AU  = radii_m * m_to_au
    first_idx = sim.N

    for i in range(n_asteroids):
        sim.add(
            m=mass_Msun[i],
            r=radii_AU[i],
            x=mars_pos[0], y=mars_pos[1], z=mars_pos[2],
            vx=vx[i], vy=vy[i], vz=vz[i],
        )

    # ── 11. DataFrame z pełnymi metadanymi ─────────────────────
    asteroid_df = pd.DataFrame({
        "sim_index":     np.arange(first_idx, first_idx + n_asteroids),
        "radius_m":      radii_m,
        "radius_AU":     radii_AU,
        "density_kg_m3": rho_arr,
        "rock_type":     rock_names,
        "albedo":        albedo_arr,
        "mass_kg":       mass_kg,
        "mass_Msun":     mass_Msun,
        "beta":          beta_arr,
        "Q_pr":          Q_pr_arr,
        "nearest_star":  nearest_star_idx,
        "v_ejecta_kms":  velocities_kms,
        "spin_period_h": spin_periods_h,
        "obliquity_deg": obliquities_deg,
        "spin_axis_x":   spin_axes[:, 0],
        "spin_axis_y":   spin_axes[:, 1],
        "spin_axis_z":   spin_axes[:, 2],
    })

    # ── Podsumowanie ────────────────────────────────────────────
    print(f"Dodano {n_asteroids} asteroid (indeksy {first_idx}–{first_idx + n_asteroids - 1})")
    print(f"  Promienie:        {radii_m.min():.4f} – {radii_m.max():.2f} m")
    print(f"  Prędkości ejecta: {velocities_kms.min():.2f} – {velocities_kms.max():.2f} km/s")
    print(f"  Beta (gwiazda #{nearest_star_idx}): {beta_arr.min():.2e} – {beta_arr.max():.2e}")
    print(f"  Typy skał:        {pd.Series(rock_names).value_counts().to_dict()}")
    print(f"  Spin period:      {spin_periods_h.min():.1f} – {spin_periods_h.max():.1f} h")
    print(f"  Cząstek łącznie:  {sim.N}")

    return asteroid_df, first_idx

## Test wybuchu

In [57]:
asteroid_df, first_ast_idx = create_mars_impact(
    sim,
    n_asteroids=200,
    cone_half_angle=55.0,
    v_min_kms=5.03,
    v_max_kms=25.0,
    alpha_v=2.5,
    R_min_m=0.001,
    R_max_m=10.0,
    q_size=1.5,
    spin_period_range=(2.0, 20.0),
    obliquity_range=(0.0, 180.0),
    size_velocity_corr=True,
    star_indices=[0] + list(range(9, 59)),
    seed=42,
)

asteroid_df.head(10)

Dodano 200 asteroid (indeksy 59–258)
  Promienie:        0.0010 – 3.25 m
  Prędkości ejecta: 5.07 – 22.98 km/s
  Beta (gwiazda #0): 6.17e-08 – 4.04e-04
  Typy skał:        {'basalt': 102, 'chondrite': 58, 'ice_rich': 40}
  Spin period:      2.0 – 19.9 h
  Cząstek łącznie:  259


,sim_index,radius_m,radius_AU,density_kg_m3,rock_type,albedo,mass_kg,mass_Msun,beta,Q_pr,nearest_star,v_ejecta_kms,spin_period_h,obliquity_deg,spin_axis_x,spin_axis_y,spin_axis_z
0,59,0.657160,4.392846e-12,3000.0,basalt,0.07,3566.349567,1.793569e-27,3.048644e-07,1.046667,0,5.095569,7.787822,109.298057,0.230267,-0.972924,-0.019915
1,60,0.384070,2.567350e-12,3000.0,basalt,0.07,711.936676,3.580432e-28,5.216361e-07,1.046667,0,5.195039,15.478774,76.496289,0.997239,0.066073,0.033905
2,61,0.031848,2.128897e-13,3000.0,basalt,0.07,0.405929,2.041476e-31,6.290686e-06,1.046667,0,5.474271,16.425010,66.597186,0.242896,0.396875,-0.885151
3,62,0.189961,1.269814e-12,3500.0,chondrite,0.15,100.497037,5.054141e-29,9.500580e-07,1.100000,0,5.243881,10.842634,75.171473,-0.914322,0.114291,-0.388526
4,63,0.001478,9.880457e-15,3000.0,basalt,0.07,0.000041,2.040851e-35,1.355426e-04,1.046667,0,12.030321,18.074558,21.311835,0.556265,-0.133626,-0.820191
5,64,0.020485,1.369326e-13,3000.0,basalt,0.07,0.108021,5.432509e-32,9.780160e-06,1.046667,0,5.523213,4.590357,143.055101,-0.417521,-0.905391,-0.077091
6,65,0.003492,2.334211e-14,3000.0,basalt,0.07,0.000535,2.690919e-34,5.737366e-05,1.046667,0,7.807293,17.823238,150.789034,-0.160384,-0.361374,0.918524
7,66,0.696480,4.655681e-12,3000.0,basalt,0.07,4245.568173,2.135157e-27,2.876534e-07,1.046667,0,5.085161,3.729714,51.191484,0.488383,-0.069431,-0.869863
8,67,0.071678,4.791387e-13,1500.0,ice_rich,0.40,2.313875,1.163681e-30,6.765118e-06,1.266667,0,5.422293,4.763459,54.683391,0.696622,0.626742,0.349158
9,68,0.001312,8.767574e-15,3000.0,basalt,0.07,0.000028,1.425997e-35,1.527472e-04,1.046667,0,13.930981,11.609120,157.380166,0.389328,-0.462058,0.796822


## Funkcja na usunięcie asteroid

In [43]:
def remove_all_asteroids(sim, n_permanent=59):
    """Usuwa wszystkie cząstki poza stałymi (Słońce + planety + gwiazdy Gaia).
    
    Domyślnie n_permanent=59: 1 Słońce + 8 planet + 50 gwiazd.
    """
    n_before = sim.N
    for i in range(sim.N - 1, n_permanent - 1, -1):
        sim.remove(i)
    print(f"Usunięto {n_before - sim.N} asteroid. Pozostało {sim.N} cząstek.")


# Wywołanie:
remove_all_asteroids(sim)

Usunięto 400 asteroid. Pozostało 59 cząstek.


## Ciśnienie promieniowania (wywołanie działa jeżeli istnieją asteroidy)

In [58]:
import reboundx
import astropy.units as u
from astropy.constants import c as c_light

# 1. Inicjalizacja REBOUNDx
rebx = reboundx.Extras(sim)

# 2. Załadowanie efektu radiation_forces
rf = rebx.load_force("radiation_forces")
rebx.add_force(rf)

# 3. Prędkość światła w jednostkach symulacji (AU/yr)
rf.params["c"] = c_light.to(u.AU / u.yr).value

# 4. Przypisanie beta do każdej asteroidy
for _, row in asteroid_df.iterrows():
    sim.particles[int(row["sim_index"])].params["beta"] = row["beta"]

print(f"Radiation forces aktywne. c = {rf.params['c']:.2f} AU/yr")
print(f"Beta przypisane {len(asteroid_df)} asteroidom.")

Radiation forces aktywne. c = 63241.08 AU/yr
Beta przypisane 200 asteroidom.


/home/kdarowski/Hack-4-sages/.venv/lib/python3.12/site-packages/rebound/simulation.py:259: RuntimeWarning: REBOUNDx overwrites sim->additional_forces, sim->pre_timestep_modifications and sim->post_timestep_modifications whenever forces or operators that use them get added.  If you want to use REBOUNDx together with your own custom functions that use these callbacks, you should add them through REBOUNDx.  See https://github.com/dtamayo/reboundx/blob/master/ipython_examples/Custom_Effects.ipynb for a tutorial.
  warnings.warn(msg[1:], RuntimeWarning)


## Moduł/funkcja promieniowania kosmicznego (na podstawie plików maksa i zofii)

## Co jeszcze:
- wybrać Integrator i zmianę czasu
- pakiet ATM - do np. policzenia temperatury w środku asteroidy (https://github.com/moeyensj/atm)
- pakiet Amuse - "wzbogacenie"
- promieniowanie kosmiczne, promieniowanie własne, hydroliza, erozja
- efekt końcowy (przechwycenie)
- zmienić parametry skał w funkcji wybuchu!!!

## Ustalić:
- albedo, gęstość i skład izotopów radioaktywnych w skałach